# 23 · Sentence-Window Retrieval

**Idea:** a core tension in chunking — small chunks match precisely but read out of context; large chunks give context but dilute the match. Sentence-window retrieval gets both: index one *sentence* at a time (precise matching), but at answer time return that sentence plus its neighbours (the "window") so the generator sees full context. Also called **small-to-big**: retrieve small, return big.

> Embeds sentences with Ollama. No cloud keys required.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

Build the retriever with `window=1` (one sentence on each side) and feed it the corpus. It explodes every chunk into per-sentence units internally.

In [ ]:
from rag_lab import load_markdown, SentenceWindowRetriever

swr = SentenceWindowRetriever(window=1, provider='ollama')
swr.add(load_markdown())
print(f'indexed {len(swr)} individual sentences')

Search matches on a *single* sentence (precise), then the hit is widened to its window. `meta['matched']` shows what actually matched vs. the fuller text returned.

In [ ]:
query = 'what does reranking do to a first-stage retrieval?'
for h in swr.search(query, k=2):
    print(f'score {h.score:.3f}  [{h.chunk.source}]')
    print('  matched  :', h.chunk.meta['matched'])
    print('  returned :', h.chunk.text, '\n')

Widen the window to `window=2` — the same precise match now returns even more surrounding context for the generator.

In [ ]:
wide = SentenceWindowRetriever(window=2, provider='ollama')
wide.add(load_markdown())
top = wide.search(query, k=1)[0]
print('window=2 returned text:\n', top.chunk.text)

**Takeaway:** decouple the *matching* unit from the *returned* unit. Index sentences for pinpoint retrieval, return windows for coherent generation — you no longer have to pick one chunk size for both jobs. The window width is a cheap dial on how much surrounding context the LLM sees.

Next up → `24_ann_indexing.ipynb`, where we open the black box of how vector stores search fast at scale.